# 3D Glaucoma — MONAI Encoder-only @ 200³ (Harvard-GF streamed from HF)

**Method scan** on 200×200×200 OCT volumes (Harvard-GF, 3,300 scans from Hugging Face).

- Model: **MONAI UNet encoder only (decoder removed)** + AdaptiveAvgPool + classification head — matches the repo's `Simple3DCNN` design philosophy.
- Hardware: Colab **A100 80GB**, bf16 AMP, **batch 2 + grad-accum** (effective batch 16), **mmap data** (200³ ≈ 26 GB total).
- Data built once from `harvardairobotics/Harvard-GF` (`Dataset/dataset.zip` + `ReadMe/data_summary.csv`) into consolidated `.npy` arrays, then mmap-loaded.

Run cells in order. First run downloads ~15-20 GB and builds arrays (~10-20 min). Sweep = 8 models × 20 epochs.


In [ ]:
!pip -q install monai umap-learn
!pip -q install hf-transfer huggingface_hub datasets


In [ ]:
# HF token (Colab Secrets -> HF_TOKEN -> hf_xxx) + Drive for saving results
from google.colab import drive, userdata
import os
drive.mount("/content/drive")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "1")


In [ ]:
import os, io, json, time, zipfile
from pathlib import Path
import numpy as np
import torch, torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from monai.networks import nets
from monai.networks.blocks import UnetBasicBlock
from monai.transforms import (Compose, RandFlip, RandRotate, RandScaleIntensity,
                              RandShiftIntensity, RandGaussianNoise)

# ---- device / AMP ----
device = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = True
_bf16 = device == "cuda" and torch.cuda.is_bf16_supported()
amp_dtype = torch.bfloat16 if _bf16 else torch.float16
SEED = 42


def to_device_normalize(x, y):
    x = x.to(device, non_blocking=True).float().div_(255.0)   # uint8 -> [0,1] on GPU
    y = y.to(device, non_blocking=True)
    return x, y


In [ ]:
# ================== 200³ data, streamed from Hugging Face ==================
# harvardairobotics/Harvard-GF  (3,300 scans; 2100 / 300 / 900 train/val/test)
HF_REPO  = "harvardairobotics/Harvard-GF"
ZIP_FILE = "Dataset/dataset.zip"       # per-scan .npz with key 'oct_bscans' (200³ uint8)
CSV_FILE = "ReadMe/data_summary.csv"   # columns: filename,glaucoma(yes/no),use(training/validation/test)
DATA_DIR = "/content/glaucoma_hf_200"  # consolidated .npy arrays land here (cached on disk)
SPLITS   = ("Training", "Validation", "Test")

RESOLUTION  = 200                      # store arrays at this size (resize 200->R if R != 200);
                                       # pick a multiple of 32 for SwinUNETR (e.g. 128/192)
BATCH_SIZE   = 2                       # 200³ -> tiny per-step batch
GRAD_ACCUM   = 8                       # effective batch = 2 * 8 = 16
CACHE_IN_RAM = False                   # 200³ = ~26 GB total -> mmap, never hold in RAM
NUM_WORKERS  = max(2, os.cpu_count() or 2)

SPLIT_ALIAS = {"training": "Training", "validation": "Validation", "valid": "Validation",
               "test": "Test", "testing": "Test"}


def download_hf(filename):
    from huggingface_hub import hf_hub_download
    print(f"[data] downloading {HF_REPO}/{filename} ...", flush=True)
    return hf_hub_download(repo_id=HF_REPO, filename=filename, repo_type="dataset")


def build_200_data():
    """Stream Harvard-GF -> per-split 200³ .npy arrays (once, low RAM, disk cached)."""
    if all(os.path.isfile(os.path.join(DATA_DIR, f"{s}_volumes.npy")) for s in SPLITS):
        print(f"[data] already built at {DATA_DIR}")
        return
    os.makedirs(DATA_DIR, exist_ok=True)
    csv_path = download_hf(CSV_FILE)
    zip_path = download_hf(ZIP_FILE)

    import csv
    meta = {}
    with open(csv_path, newline="") as fh:
        for r in csv.DictReader(fh):
            split = SPLIT_ALIAS.get((r["use"] or "").strip().lower())
            if split is None:
                continue
            gl = 1 if str(r["glaucoma"]).strip().lower() in ("yes", "1", "true") else 0
            meta[Path(r["filename"]).stem] = (split, gl)
    print(f"[data] {len(meta)} labeled samples from CSV")

    with zipfile.ZipFile(zip_path) as zf:
        names = [n for n in zf.namelist() if n.endswith(".npz")]
        counts = {s: 0 for s in SPLITS}
        for n in names:
            m = meta.get(Path(n).stem)
            if m:
                counts[m[0]] += 1
    print("[data] zip-matched counts:", counts)

    vols, labels = {}, {}
    for s in SPLITS:
        vp = os.path.join(DATA_DIR, f"{s}_volumes.npy")
        vols[s] = np.lib.format.open_memmap(vp, mode="w+", dtype=np.uint8,
                                            shape=(counts[s], 1, RESOLUTION, RESOLUTION, RESOLUTION))
        labels[s] = np.zeros(counts[s], dtype=np.int64)

    filled = {s: 0 for s in SPLITS}
    with zipfile.ZipFile(zip_path) as zf:
        for n in names:
            m = meta.get(Path(n).stem)
            if not m:
                continue
            split, label = m
            raw = np.load(io.BytesIO(zf.read(n)))["oct_bscans"]      # (200,200,200) uint8
            if RESOLUTION != 200:
                t = torch.from_numpy(raw).float().div_(255.0).unsqueeze(0).unsqueeze(0)
                t = torch.nn.functional.interpolate(
                    t, size=(RESOLUTION,) * 3, mode="trilinear", align_corners=False)
                raw = (t.squeeze(0, 1).clamp(0, 1) * 255).round().numpy().astype(np.uint8)
            vols[split][filled[split]] = raw[None]                   # -> (1,R,R,R)
            labels[split][filled[split]] = label
            filled[split] += 1
    for s in SPLITS:
        vols[s].flush()
        np.save(os.path.join(DATA_DIR, f"{s}_labels.npy"), labels[s])
        print(f"[data] {s}: {filled[s]} volumes ({(counts[s] * 8 / 1e9):.1f} GB)")
    with open(os.path.join(DATA_DIR, "manifest.json"), "w") as fh:
        json.dump({"source": HF_REPO, "size_name": "200", "store_shape": [1, 200, 200, 200],
                   "splits": {s: {"built_n": filled[s]} for s in SPLITS}}, fh, indent=2)


class OCTMemmapDataset(Dataset):
    """Consolidated {split}_volumes.npy is (N,1,200,200,200) uint8; labels (N,) int64."""

    def __init__(self, data_dir, split, cache_in_ram=False):
        self.labels = np.load(os.path.join(data_dir, f"{split}_labels.npy"))
        vp = os.path.join(data_dir, f"{split}_volumes.npy")
        self.volumes = np.load(vp) if cache_in_ram else np.load(vp, mmap_mode="r")

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        x = torch.from_numpy(np.ascontiguousarray(self.volumes[idx]).copy())  # uint8 (1,200,200,200), writable
        y = torch.tensor(int(self.labels[idx]), dtype=torch.long)
        return x, y


def build_loaders():
    build_200_data()
    counts = {s: len(np.load(os.path.join(DATA_DIR, f"{s}_labels.npy"))) for s in SPLITS}
    print("[data] counts:", counts)
    workers = 0 if CACHE_IN_RAM else NUM_WORKERS
    kw = dict(batch_size=BATCH_SIZE, num_workers=workers, pin_memory=(device == "cuda"))
    if workers > 0:
        kw.update(persistent_workers=True, prefetch_factor=4)
    train_ds = OCTMemmapDataset(DATA_DIR, "Training",   cache_in_ram=CACHE_IN_RAM)
    val_ds   = OCTMemmapDataset(DATA_DIR, "Validation", cache_in_ram=CACHE_IN_RAM)
    test_ds  = OCTMemmapDataset(DATA_DIR, "Test",       cache_in_ram=CACHE_IN_RAM)
    g = torch.Generator(); g.manual_seed(SEED)
    train_loader = DataLoader(train_ds, shuffle=True, generator=g, **kw)
    val_loader   = DataLoader(val_ds,   shuffle=False, **kw)
    test_loader  = DataLoader(test_ds,  shuffle=False, **kw)
    return train_loader, val_loader, test_loader


train_loader, val_loader, test_loader = build_loaders()


In [ ]:
# ---- conservative 3D augmentation (training only; keeps OCT structure) ----
def make_train_transform():
    return Compose([
        RandFlip(prob=0.5, spatial_axis=1),                    # flip H
        RandFlip(prob=0.5, spatial_axis=2),                    # flip W
        RandRotate(range_x=0.10, range_y=0.10, range_z=0.10,   # ±~6°
                   prob=0.5, mode="bilinear", padding_mode="zeros", keep_size=True),
        RandScaleIntensity(factors=0.10, prob=0.5),            # *(1±0.1)
        RandShiftIntensity(offsets=10.0, prob=0.5),            # ±10 on [0,255]
        RandGaussianNoise(prob=0.3, std=5.0),
    ])


class TrainAug:
    """Wraps the memmap train split and applies MONAI transforms on-the-fly."""

    def __init__(self, ds, transform):
        self.ds, self.tf = ds, transform

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, i):
        x, y = self.ds[i]
        x = torch.as_tensor(self.tf(x)[0])      # MONAI may drop the singleton channel
        if x.ndim == 3:
            x = x.unsqueeze(0)                  # restore (1,200,200,200)
        return x, y


g = torch.Generator(); g.manual_seed(SEED)
workers = 0 if CACHE_IN_RAM else NUM_WORKERS
train_loader = DataLoader(TrainAug(train_loader.dataset, make_train_transform()),
                          batch_size=BATCH_SIZE, shuffle=True, generator=g,
                          num_workers=workers, pin_memory=(device == "cuda"),
                          persistent_workers=workers > 0, prefetch_factor=4 if workers > 0 else None)


In [ ]:
class UNetEncoder3D(nn.Module):
    """MONAI UNet ENCODER ONLY (decoder removed) + AdaptiveAvgPool + classification head."""

    def __init__(self, in_channels=1, num_classes=2, features=(32, 64, 128, 256),
                 strides=(2, 2, 2), num_res_units=2, norm="batch", dropout=0.0):
        super().__init__()
        self.features = tuple(features)
        strides = tuple(strides)
        if len(strides) != len(features) - 1:      # one pool between consecutive levels
            strides = (2,) * (len(features) - 1)
        self.down_blocks = nn.ModuleList()
        self.down_samples = nn.ModuleList()
        cin = in_channels
        for i, f in enumerate(features):
            self.down_blocks.append(UnetBasicBlock(
                3, cin, f, kernel_size=3, stride=1,
                norm_name=norm, act_name="relu", dropout=dropout))
            cin = f
            if i < len(features) - 1:
                self.down_samples.append(nn.MaxPool3d(strides[i], stride=strides[i]))
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool3d(1),
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(features[-1], num_classes),
        )

    def forward(self, x):
        for blk, samp in zip(self.down_blocks, self.down_samples + [nn.Identity()]):
            x = blk(x)
            x = samp(x)
        return self.head(x)


class Simple3DCNN(nn.Module):
    """Project reference baseline (models/glaucoma/model.py, unchanged)."""

    def __init__(self, in_channels=1, num_classes=2, dropout=0.3):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv3d(in_channels, 16, kernel_size=3, padding=1), nn.BatchNorm3d(16),
            nn.ReLU(), nn.MaxPool3d(2),
            nn.Conv3d(16, 32, kernel_size=3, padding=1), nn.BatchNorm3d(32),
            nn.ReLU(), nn.MaxPool3d(2),
            nn.Conv3d(32, 64, kernel_size=3, padding=1), nn.BatchNorm3d(64),
            nn.ReLU(), nn.MaxPool3d(2),
            nn.Conv3d(64, 128, kernel_size=3, padding=1), nn.BatchNorm3d(128),
            nn.ReLU(), nn.AdaptiveAvgPool3d(1),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(128, 64), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(64, num_classes))

    def forward(self, x):
        return self.classifier(self.features(x))


@torch.no_grad()
def eval_acc(model, loader):
    model.eval(); c = t = 0
    for x, y in loader:
        x, y = to_device_normalize(x, y)
        with torch.autocast("cuda", dtype=amp_dtype, enabled=(USE_AMP and device == "cuda")):
            logits = model(x)
        c += (logits.argmax(1) == y).sum().item(); t += y.numel()
    return c / t


In [ ]:
# ====== SOTA config: set `name` to the sweep WINNER (enc-32 default) ======
SOTA_CFG = {
    "name": "enc-32",                 # <- overwrite with sweep_results_200.json WINNER
    "features": (32, 64, 128, 256),
    "num_res_units": 2,
    "epochs": 30,
    "batch_size": BATCH_SIZE,
    "grad_accum_steps": GRAD_ACCUM,   # effective batch = 16
    "lr": 1e-4 * ((BATCH_SIZE * GRAD_ACCUM) / 4) ** 0.5,
    "weight_decay": 0.01,
    "warmup_ratio": 0.05,
    "max_grad_norm": 1.0,
    "patience": 8,
    "output_dir": "/content/drive/MyDrive/MasterBKDN/Thesis/sota_200",
}


def build_sota():
    if "features" in SOTA_CFG:
        return UNetEncoder3D(in_channels=1, num_classes=2,
                             features=SOTA_CFG["features"],
                             num_res_units=SOTA_CFG["num_res_units"])
    return Simple3DCNN(in_channels=1, num_classes=2)


# ====== train (pipeline/train.py practices: warmup+cosine, grad-accum, clip, AMP) ======
import math
os.makedirs(SOTA_CFG["output_dir"], exist_ok=True)

torch.manual_seed(SEED)
model = build_sota().to(device)
crit = nn.CrossEntropyLoss()
steps_per_epoch = math.ceil(len(train_loader) / SOTA_CFG["grad_accum_steps"])
total_steps = steps_per_epoch * SOTA_CFG["epochs"]
opt = torch.optim.AdamW(model.parameters(), lr=SOTA_CFG["lr"], weight_decay=SOTA_CFG["weight_decay"])
warmup = int(total_steps * SOTA_CFG["warmup_ratio"])
if warmup > 0:
    sched = torch.optim.lr_scheduler.SequentialLR(
        opt,
        [torch.optim.lr_scheduler.LinearLR(opt, start_factor=0.01, total_iters=warmup),
         torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=total_steps - warmup)],
        milestones=[warmup])
else:
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=total_steps)
scaler = torch.amp.GradScaler("cuda",
            enabled=(USE_AMP and device == "cuda" and amp_dtype == torch.float16))

best_val, bad, gstep, t0 = 0.0, 0, 0, time.time()
for ep in range(SOTA_CFG["epochs"]):
    model.train(); opt.zero_grad(set_to_none=True)
    for i, (x, y) in enumerate(train_loader):
        x, y = to_device_normalize(x, y)
        with torch.autocast("cuda", dtype=amp_dtype, enabled=(USE_AMP and device == "cuda")):
            loss = crit(model(x), y) / SOTA_CFG["grad_accum_steps"]
        scaler.scale(loss).backward()
        if (i + 1) % SOTA_CFG["grad_accum_steps"] == 0:
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), SOTA_CFG["max_grad_norm"])
            scaler.step(opt); scaler.update(); sched.step()
            opt.zero_grad(set_to_none=True); gstep += 1
    va = eval_acc(model, val_loader)
    print(f"ep{ep+1:02d} val={va:.4f} (best {best_val:.4f}) | {(time.time()-t0)/60:.1f} min | lr={sched.get_last_lr()[0]:.2e}")
    if va > best_val:
        best_val, bad = va, 0
        torch.save(model.state_dict(), os.path.join(SOTA_CFG["output_dir"], "best_model.pt"))
    else:
        bad += 1
        if bad >= SOTA_CFG["patience"]:
            print("[early-stop] no improvement on val")
            break

# ====== final held-out test eval ======
@torch.no_grad()
def test_eval():
    m = build_sota().to(device)
    m.load_state_dict(torch.load(os.path.join(SOTA_CFG["output_dir"], "best_model.pt"),
                                 map_location=device))
    return eval_acc(m, test_loader)

test_acc = test_eval()
print(f"[test] acc={test_acc:.4f}")
with open(os.path.join(SOTA_CFG["output_dir"], "test_results.json"), "w") as fh:
    json.dump({"model": SOTA_CFG["name"], "test_acc": test_acc, "best_val": best_val}, fh, indent=2)
print(f"[saved] best_model.pt + test_results.json -> {SOTA_CFG['output_dir']}")


In [ ]:
# ====== XAI helpers (test split, in-memory, no aug) ======
import matplotlib.pyplot as plt

XAI_DIR = os.path.join(SOTA_CFG["output_dir"], "xai")
os.makedirs(XAI_DIR, exist_ok=True)

test_ds = OCTMemmapDataset(DATA_DIR, "Test", cache_in_ram=False)


def load_sample(ds, idx):
    """uint8 (1,200,200,200) -> normalized (1,1,200,200,200) on GPU + label."""
    x, y = ds[idx]
    x = x.float().div_(255.0).unsqueeze(0).to(device)
    return x, int(y)


def overlay_slices(vol, cam, title, fname, axis=0, fracs=(0.4, 0.5, 0.6)):
    """vol/cam: (D,H,W) in [0,1]. Blend 3 mid slices along `axis` and save."""
    fig, axes = plt.subplots(1, len(fracs), figsize=(5 * len(fracs), 5))
    for ax, fr in zip(axes, fracs):
        idx = int(vol.shape[axis] * fr)
        if axis == 0:
            sl, cmap = vol[idx], cam[idx]
        elif axis == 1:
            sl, cmap = vol[:, idx, :], cam[:, idx, :]
        else:
            sl, cmap = vol[:, :, idx], cam[:, :, idx]
        ax.imshow(sl, cmap="gray")
        ax.imshow(cmap, cmap="jet", alpha=0.5, vmin=0.0, vmax=1.0)
        ax.set_title(f"{'D' if axis==0 else 'H' if axis==1 else 'W'}={idx}")
        ax.axis("off")
    fig.suptitle(title)
    fig.savefig(os.path.join(XAI_DIR, fname), dpi=120, bbox_inches="tight")
    plt.show()


# load trained best model
model = build_sota().to(device)
model.load_state_dict(torch.load(os.path.join(SOTA_CFG["output_dir"], "best_model.pt"),
                                 map_location=device))
model.eval()


In [ ]:
# ====== A. 3D Grad-CAM (on the last encoder block) ======
def _target_block(model):
    """Last encoder conv block: MONAI UNetEncoder3D -> down_blocks[-1]; Simple3DCNN -> last Conv3d."""
    if hasattr(model, "down_blocks"):
        return model.down_blocks[-1]
    return [m for m in model.features.modules() if isinstance(m, nn.Conv3d)][-1]


def grad_cam(model, x, target=None):
    """Returns (cam 200³ in [0,1], pred_class, pred_prob) for the predicted class."""
    model.eval()
    if target is None:
        target = _target_block(model)
    store = {}
    handle = target.register_forward_hook(lambda m, i, o: store.__setitem__("a", o))
    with torch.autocast("cuda", dtype=amp_dtype, enabled=(device == "cuda")):
        out = model(x)
    handle.remove()
    cls = int(out.argmax(1))
    score = out[0, cls]
    a = store["a"]
    grad = torch.autograd.grad(score, a)[0]
    w = grad.mean(dim=(2, 3, 4), keepdim=True)             # channel weights
    cam = (w * a).sum(1, keepdim=True).relu()              # (1,1,c,h,w)
    cam = F.interpolate(cam, size=x.shape[2:], mode="trilinear", align_corners=False)
    cam = cam.squeeze(0, 1).detach().cpu()
    cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    return cam, cls, torch.softmax(out, 1)[0, cls].item()


# one positive + one negative test sample, overlaid on axial/sagittal/coronal
for target_label in (0, 1):
    idx = next(i for i in range(len(test_ds)) if int(test_ds.labels[i]) == target_label)
    x, y = load_sample(test_ds, idx)
    cam, cls, p = grad_cam(model, x)
    print(f"sample idx={idx} true={y} pred={cls} p={p:.3f}")
    vol = x.squeeze(0, 1).cpu().numpy()
    for axis, aname in ((0, "axial"), (1, "sagittal"), (2, "coronal")):
        overlay_slices(vol, cam.numpy(),
                       f"Grad-CAM true={y} pred={cls} p={p:.2f} ({aname})",
                       f"gradcam_label{target_label}_{aname}.png", axis=axis)


In [ ]:
# ====== B. Integrated Gradients (zero baseline) ======
def integrated_gradients(model, x, steps=20, baseline=None):
    """Per-voxel attribution (abs) for the predicted class, [0,1]. 20 fwd+bwd passes."""
    model.eval()
    if baseline is None:
        baseline = torch.zeros_like(x)
    with torch.autocast("cuda", dtype=amp_dtype, enabled=(device == "cuda")):
        out0 = model(baseline)
    cls = int(out0.argmax(1))
    path = torch.linspace(0, 1, steps, device=x.device)
    grads = []
    for t in path:
        p = (baseline + t * (x - baseline)).detach().requires_grad_(True)
        with torch.autocast("cuda", dtype=amp_dtype, enabled=(device == "cuda")):
            o = model(p)
        g = torch.autograd.grad(o[0, cls], p)[0]
        grads.append(g.detach())
    ig = (x - baseline) * torch.stack(grads).mean(0)
    ig = ig.squeeze(0, 1).abs().cpu()
    ig = (ig - ig.min()) / (ig.max() - ig.min() + 1e-8)
    return ig, cls


for target_label in (0, 1):
    idx = next(i for i in range(len(test_ds)) if int(test_ds.labels[i]) == target_label)
    x, y = load_sample(test_ds, idx)
    ig, cls = integrated_gradients(model, x, steps=20)
    print(f"sample idx={idx} true={y} pred={cls}")
    vol = x.squeeze(0, 1).cpu().numpy()
    for axis, aname in ((0, "axial"), (1, "sagittal"), (2, "coronal")):
        overlay_slices(vol, ig.numpy(),
                       f"IntegratedGradients true={y} pred={cls} ({aname})",
                       f"ig_label{target_label}_{aname}.png", axis=axis)


In [ ]:
# ====== C. Occlusion sensitivity (sliding 40³ patches; slowest method) ======
@torch.no_grad()
def occlusion_sensitivity(model, x, cube=40, stride=40, fill=0.0):
    """Prediction drop of the predicted class when 40³ cubes are masked (5³=125 fwd passes)."""
    model.eval()
    _, _, D, H, W = x.shape
    with torch.autocast("cuda", dtype=amp_dtype, enabled=(device == "cuda")):
        base = torch.softmax(model(x), 1)
    target = int(base.argmax(1)); base_p = base[0, target].item()
    heat = np.zeros((D, H, W)); cnt = np.zeros((D, H, W))
    for d in range(0, D, stride):
        for h in range(0, H, stride):
            for w in range(0, W, stride):
                xm = x.clone()
                xm[:, :, d:d+cube, h:h+cube, w:w+cube] = fill
                with torch.autocast("cuda", dtype=amp_dtype, enabled=(device == "cuda")):
                    p = torch.softmax(model(xm), 1)[0, target].item()
                heat[d:d+cube, h:h+cube, w:w+cube] += (base_p - p)
                cnt[d:d+cube, h:h+cube, w:w+cube] += 1
    cnt[cnt == 0] = 1
    heat = heat / cnt
    hm = heat - heat.min()
    heat = hm / (hm.max() + 1e-8)
    return heat, base_p


idx = next(i for i in range(len(test_ds)) if int(test_ds.labels[i]) == 1)
x, y = load_sample(test_ds, idx)
heat, base_p = occlusion_sensitivity(model, x, cube=40, stride=40)
print(f"sample idx={idx} true={y} | base P(pred)={base_p:.3f}")
vol = x.squeeze(0, 1).cpu().numpy()
for axis, aname in ((0, "axial"), (1, "sagittal"), (2, "coronal")):
    overlay_slices(vol, heat,
                   f"Occlusion-sensitivity true={y} (drop map, {aname})",
                   f"occlusion_{aname}.png", axis=axis)


In [ ]:
# ====== F. UMAP of penultimate embeddings (before the classifier head) ======
def penultimate(model, x):
    """Embedding = output of the pooling layer (hook on head[0] / last AdaptiveAvgPool3d)."""
    if hasattr(model, "head"):
        target = model.head[0]
    else:
        target = [m for m in model.features.modules() if isinstance(m, nn.AdaptiveAvgPool3d)][-1]
    pooled = {}
    handle = target.register_forward_hook(lambda m, i, o: pooled.__setitem__("e", o.detach().flatten(1)))
    with torch.autocast("cuda", dtype=amp_dtype, enabled=(device == "cuda")):
        model(x)
    handle.remove()
    return pooled["e"]


@torch.no_grad()
def collect_embeddings(model, ds, max_n=500, seed=SEED):
    rng = np.random.default_rng(seed)
    idxs = rng.choice(len(ds), size=min(max_n, len(ds)), replace=False)
    Es, Ys = [], []
    for i in idxs:
        x, y = load_sample(ds, int(i))
        Es.append(penultimate(model, x).cpu())
        Ys.append(y)
    return torch.cat(Es).numpy(), np.array(Ys)


from umap import UMAP
E, Y = collect_embeddings(model, test_ds, max_n=500)
emb = UMAP(n_neighbors=15, min_dist=0.1, random_state=SEED).fit_transform(E)
fig, ax = plt.subplots(figsize=(7, 6))
for c, lab, col in ((0, "no_glaucoma", "tab:blue"), (1, "glaucoma", "tab:red")):
    m = Y == c
    ax.scatter(emb[m, 0], emb[m, 1], s=12, c=col, label=f"{lab} (n={m.sum()})", alpha=0.7)
ax.legend(); ax.set_title("UMAP of penultimate embeddings (test, 500 samples)")
fig.savefig(os.path.join(XAI_DIR, "umap_embeddings.png"), dpi=120, bbox_inches="tight")
plt.show()


In [ ]:
# ====== G. Confusion matrix + Grad-CAM on mistakes (FP / FN) ======
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

all_pred, all_y = [], []
for x, y in test_loader:
    x, y = to_device_normalize(x, y)
    with torch.autocast("cuda", dtype=amp_dtype, enabled=(USE_AMP and device == "cuda")):
        logits = model(x)
    all_pred.extend(logits.argmax(1).cpu().tolist())
    all_y.extend(y.cpu().tolist())

cm = confusion_matrix(all_y, all_pred)
print(classification_report(all_y, all_pred, target_names=["no_glaucoma", "glaucoma"]))
ConfusionMatrixDisplay(cm, display_labels=["no_glaucoma", "glaucoma"]).plot(cmap="Blues")
plt.title("Confusion matrix (test)")
plt.savefig(os.path.join(XAI_DIR, "confusion_matrix.png"), dpi=120, bbox_inches="tight")
plt.show()

# Grad-CAM on the first few mistakes (test_loader is un-shuffled over test_ds -> same indices)
fp_idx = [i for i in range(len(all_y)) if all_pred[i] == 1 and all_y[i] == 0]
fn_idx = [i for i in range(len(all_y)) if all_pred[i] == 0 and all_y[i] == 1]
print(f"False positives: {len(fp_idx)} | False negatives: {len(fn_idx)}")
for kind, group in (("FP", fp_idx), ("FN", fn_idx)):
    for j in group[:2]:
        x, y = load_sample(test_ds, j)
        cam, cls, p = grad_cam(model, x)
        print(f"{kind} test_idx={j} true={y} pred={cls} p={p:.3f}")
        vol = x.squeeze(0, 1).cpu().numpy()
        overlay_slices(vol, cam.numpy(), f"{kind} true={y} pred={cls} p={p:.2f}",
                       f"mistake_{kind}_{j}.png", axis=0)


In [ ]:
# ====== Interpretation summary (shortcut-learning / convergence check) ======
summary = """
================= XAI INTERPRETATION CHECKLIST =================
3D Grad-CAM / Integrated Gradients / Occlusion:
  [ ] Heatmap over retina/anatomy          -> model uses relevant signal (good)
  [ ] Heatmap over background / scanner border -> SHORTCUT LEARNING -> crop foreground, mask bg
  [ ] Heatmap over border markers/artifacts -> data leakage -> fix preprocessing
  [ ] Heatmap random / noisy               -> not converged -> check LR, labels, normalization

UMAP embeddings:
  [ ] Classes separate well                -> separable features; head/classifier may be weak
  [ ] Classes overlap completely           -> no discriminative features -> labels/data/task
  [ ] Clusters by scanner/protocol         -> domain shift / batch effect -> harmonization

Confusion matrix:
  [ ] FN > FP and FN focus on subtle cases -> low resolution / need lesion-centered sampling
  [ ] Errors spread evenly                 -> label ambiguity -> expert review / better labels
=================================================================
"""
print(summary)
with open(os.path.join(XAI_DIR, "interpretation_summary.txt"), "w") as fh:
    fh.write(summary)
print(f"[saved] all XAI figures -> {XAI_DIR}")


In [ ]:
# Done. Release the GPU immediately.
from google.colab import runtime
runtime.unassign()
